In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [17]:
torch.cuda.is_available()
df = pd.read_csv("cleaned_final.csv")
#df = pd.read_csv("RTN_test.csv")

features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
targets = ['x_error', 'y_error', 'z_error']

#features = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'days_offset']
#targets = ['r_error', 't_error', 'n_error']

X = df[features].values
y = df[targets].values

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

df_test = df.iloc[split_idx:].reset_index(drop=True)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

# X_tensor = torch.tensor(X, dtype=torch.float32)
# y_tensor = torch.tensor(y, dtype=torch.float32)
# dataset = TensorDataset(X_tensor, y_tensor)

train_dataset = TensorDataset(  
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(y_train, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32), 
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


In [18]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(7, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device=device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)

epochs = 1000
for epoch in range(epochs):
    model.train()
    ov_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        y_pred = model(batch_X)
        loss = loss_fn(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ov_loss += loss.item() * batch_X.size(0)

    epoch_loss = ov_loss / len(train_dataset)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

Epoch [100/1000], Loss: 0.3960
Epoch [200/1000], Loss: 0.2076
Epoch [300/1000], Loss: 0.1944
Epoch [400/1000], Loss: 0.1720
Epoch [500/1000], Loss: 0.1353
Epoch [600/1000], Loss: 0.1183
Epoch [700/1000], Loss: 0.1150
Epoch [800/1000], Loss: 0.1292
Epoch [900/1000], Loss: 0.1188
Epoch [1000/1000], Loss: 0.1199


In [19]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        y_pred = model(batch_X)
        all_preds.append(y_pred.cpu().numpy())
        all_targets.append(batch_y.cpu().numpy())

preds_array = np.vstack(all_preds)
targets_array = np.vstack(all_targets)

preds_km = scaler_y.inverse_transform(preds_array)
targets_km = scaler_y.inverse_transform(targets_array)

err3d_model = np.linalg.norm(preds_km - targets_km, axis=1)
err3d_raw_sgp4 = np.linalg.norm(targets_km, axis=1)

df_test['Err3D_Model'] = err3d_model
df_test['Err3D_SGP4'] = err3d_raw_sgp4

print("\n=== PODSUMOWANIE OGÓLNE 3D [km] ===")
print(f"SGP4  -> Średnia: {err3d_raw_sgp4.mean():.3f} | Mediana: {np.median(err3d_raw_sgp4):.3f} | 90th perc (ogon): {np.percentile(err3d_raw_sgp4, 90):.3f}")
print(f"Model -> Średnia: {err3d_model.mean():.3f} | Mediana: {np.median(err3d_model):.3f} | 90th perc (ogon): {np.percentile(err3d_model, 90):.3f}")

print("\n=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===")
summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({
    'Liczba_Próbek': len(g),
    'Mediana_SGP4': g['Err3D_SGP4'].median(),
    'Mediana_Model': g['Err3D_Model'].median(),
    'Średnia_SGP4': g['Err3D_SGP4'].mean(),
    'Średnia_Model': g['Err3D_Model'].mean(),
    '90perc_SGP4': np.percentile(g['Err3D_SGP4'], 90),
    '90perc_Model': np.percentile(g['Err3D_Model'], 90)
}))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(summary.round(3).to_string())


=== PODSUMOWANIE OGÓLNE 3D [km] ===
SGP4  -> Średnia: 3.777 | Mediana: 2.290 | 90th perc (ogon): 9.474
Model -> Średnia: 3.391 | Mediana: 2.590 | 90th perc (ogon): 6.989

=== BŁĄD 3D W ROZBICIU NA DAYS_OFFSET [km] ===
             Liczba_Próbek  Mediana_SGP4  Mediana_Model  Średnia_SGP4  Średnia_Model  90perc_SGP4  90perc_Model
days_offset                                                                                                    
0                    167.0         1.289          1.766         1.468          1.922        2.474         3.363
1                    166.0         1.210          1.874         1.392          1.986        2.485         3.422
2                    165.0         1.150          1.886         1.295          2.032        2.177         3.350
3                    163.0         1.256          1.988         1.365          2.130        2.146         3.670
4                    161.0         1.678          2.100         1.753          2.312        2.883         4.0

C:\Users\kozak\AppData\Local\Temp\ipykernel_23892\2958093815.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df_test.groupby('days_offset').apply(lambda g: pd.Series({


In [20]:
# from sklearn.ensemble import GradientBoostingRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error

# # Szybki test na obecnych danych
# X_train, X_test, y_train, y_test = train_test_split(X, y[:, 0], test_size=0.2, shuffle=False) # Chronologicznie!

# gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)
# gbr.fit(X_train, y_train)
# preds = gbr.predict(X_test)

# print("MAE Gradient Boosting dla osi X:", mean_absolute_error(y_test, preds))

In [21]:
N = 10  # ile ostatnich wierszy pokazać

view = pd.DataFrame({
    'target_epoch':  df_test['target_epoch'].values,
    'days_offset':   df_test['days_offset'].values,

    # prawdziwy błąd SGP4 (prawda_SP3 - SGP4), bez korekty
    'SGP4_dx': targets_km[:, 0],
    'SGP4_dy': targets_km[:, 1],
    'SGP4_dz': targets_km[:, 2],

    # korekta przewidziana przez model
    'Model_dx': preds_km[:, 0],
    'Model_dy': preds_km[:, 1],
    'Model_dz': preds_km[:, 2],

    # odległość od prawdy w 3D: bez korekty vs z korektą
    'Dist_SGP4 [km]':  np.linalg.norm(targets_km, axis=1),
    'Dist_Model [km]': np.linalg.norm(targets_km - preds_km, axis=1),
})
view['Poprawa [km]'] = view['Dist_SGP4 [km]'] - view['Dist_Model [km]']  # >0 = model pomógł

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
print(view.tail(N).round(3).to_string(index=False))

       target_epoch  days_offset  SGP4_dx  SGP4_dy  SGP4_dz  Model_dx  Model_dy  Model_dz  Dist_SGP4 [km]  Dist_Model [km]  Poprawa [km]
2025-11-16 12:00:00            3   -0.832   -0.070    0.722    -0.685     0.468     1.046           1.104            0.645         0.459
2025-11-17 12:00:00            4   -1.533   -0.065    1.140    -1.540     0.389     1.314           1.912            0.487         1.425
2025-11-18 12:00:00            5   -2.334    0.016    1.546    -2.414     0.342     1.666           2.800            0.356         2.444
2025-11-15 12:00:00            0    0.559    0.140   -0.541     0.837     0.698    -0.126           0.791            0.749         0.042
2025-11-16 12:00:00            1    0.305    0.127   -0.356     0.360     0.680     0.014           0.486            0.668        -0.182
2025-11-17 12:00:00            2   -0.033    0.182   -0.214    -0.097     0.810     0.109           0.283            0.710        -0.427
2025-11-18 12:00:00            3   -0.475

In [22]:
view = pd.DataFrame({
    'tle_epoch':   df_test['tle_epoch'].values,
    'target_epoch': df_test['target_epoch'].values,
    'days_offset': df_test['days_offset'].values,

    # surowy wynik SGP4 (pozycja, km)
    'SGP4_x': df_test['x'].values,
    'SGP4_y': df_test['y'].values,
    'SGP4_z': df_test['z'].values,

    # surowy wynik modelu (przewidziany błąd, km)
    'Model_x': preds_km[:, 0],
    'Model_y': preds_km[:, 1],
    'Model_z': preds_km[:, 2],
})
print(view.tail(10).round(3).to_string(index=False))

                 tle_epoch        target_epoch  days_offset     SGP4_x    SGP4_y     SGP4_z  Model_x  Model_y  Model_z
2025-11-13 15:19:38.832960 2025-11-16 12:00:00            3 -22556.161 23519.259 -26775.489   -0.685    0.468    1.046
2025-11-13 15:19:38.832960 2025-11-17 12:00:00            4 -22686.361 23824.754 -26395.507   -1.540    0.389    1.314
2025-11-13 15:19:38.832960 2025-11-18 12:00:00            5 -22805.514 24136.201 -26009.867   -2.414    0.342    1.666
2025-11-15 15:12:38.515968 2025-11-15 12:00:00            0 -22415.789 23219.704 -27149.081    0.837    0.698   -0.126
2025-11-15 15:12:38.515968 2025-11-16 12:00:00            1 -22557.298 23519.061 -26774.411    0.360    0.680    0.014
2025-11-15 15:12:38.515968 2025-11-17 12:00:00            2 -22687.862 23824.507 -26394.153   -0.097    0.810    0.109
2025-11-15 15:12:38.515968 2025-11-18 12:00:00            3 -22807.374 24135.913 -26008.223   -0.497    0.943    0.185
2025-11-16 17:57:22.499712 2025-11-16 12:00:00  

In [23]:
for a in [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]:
    e = np.linalg.norm(targets_km - a * preds_km, axis=1)
    print(a, e.mean().round(3), np.median(e).round(3), np.percentile(e, 90).round(3))

0 3.777 2.29 9.474
0.25 3.388 2.101 8.314
0.5 3.172 2.092 7.449
0.75 3.16 2.218 6.89
1.0 3.391 2.59 6.989
1.5 4.442 3.6 8.817
2.0 5.836 4.672 12.338


In [24]:
use = df_test['days_offset'].values >= 6
e_hyb = np.where(use, err3d_model, err3d_raw_sgp4)

d = df_test.assign(sgp4=err3d_raw_sgp4, hyb=e_hyb)
groups = {t: (g['sgp4'].values, g['hyb'].values) for t, g in d.groupby('tle_epoch')}
keys = list(groups)
rng = np.random.default_rng(0)

res = []
for _ in range(2000):
    pick = rng.choice(len(keys), len(keys))                  # losujemy całe TLE
    a = np.concatenate([groups[keys[i]][0] for i in pick])
    b = np.concatenate([groups[keys[i]][1] for i in pick])
    res.append((a.mean() - b.mean(), np.median(a) - np.median(b),
                np.percentile(a, 90) - np.percentile(b, 90)))
print(np.percentile(res, [2.5, 50, 97.5], axis=0).round(3))   # kolumny: średnia, mediana, p90; >0 = hybryda lepsza

[[0.434 0.085 1.478]
 [0.686 0.231 2.634]
 [0.94  0.393 3.493]]


In [25]:
omega = np.array([0.0, 0.0, 7.292115e-5])      # rad/s
r = df[['x','y','z']].values
v = df[['vx','vy','vz']].values
v_in = v + np.cross(omega, r)

cos = np.sum(r*v_in, axis=1) / (np.linalg.norm(r, axis=1) * np.linalg.norm(v_in, axis=1))
print(np.percentile(np.abs(cos), [5, 50, 95]).round(4))      # oczekiwane: 0.00x
print(np.percentile(np.linalg.norm(v_in, axis=1), [5, 50, 95]).round(3))   # oczekiwane: ok. 3.07

[0.0002 0.0016 0.0024]
[3.068 3.075 3.082]
